In [ ]:
import requests

# Get voting records for a specific case
response = requests.get(
    "https://oda.ft.dk/api/Afstemning",
    params={"$expand": "Stemme"} #"$filter": "sagid eq 102903",
)

if response.status_code == 200:
    try:
        data = response.json()
        print("Success:", data)
    except ValueError:
        print("Error: Response is not valid JSON")
        print("Response text:", response.text)
else:
    print("HTTP error:", response.status_code)
    print("Response text:", response.text)
# data = response.json()

Success: {'odata.metadata': 'https://oda.ft.dk/api/$metadata#Afstemning', 'value': [{'Stemme': [{'id': 53, 'typeid': 1, 'afstemningid': 1, 'aktørid': 5, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 124, 'typeid': 3, 'afstemningid': 1, 'aktørid': 12, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 177, 'typeid': 3, 'afstemningid': 1, 'aktørid': 13, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 133, 'typeid': 3, 'afstemningid': 1, 'aktørid': 17, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 64, 'typeid': 1, 'afstemningid': 1, 'aktørid': 18, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 162, 'typeid': 3, 'afstemningid': 1, 'aktørid': 23, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 35, 'typeid': 1, 'afstemningid': 1, 'aktørid': 24, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 34, 'typeid': 1, 'afstemningid': 1, 'aktørid': 28, 'opdateringsdato': '2014-09-09T09:05:59.653'}, {'id': 111, 'typeid': 3, 'afstemningid': 1, 'aktørid': 33, 'o

In [ ]:
import requests
import pandas as pd
from collections import defaultdict

class PartyVotingAnalyzer:
    def __init__(self):
        self.base_url = "https://oda.ft.dk/api/"
        
    def get_voting_session_with_votes(self, voting_id):
        """Retrieve a complete voting session with all individual votes"""
        params = {
            '$expand': 'Stemme/Aktør',
            '$filter': f'id eq {voting_id}'
        }
        url = f"{self.base_url}Afstemning"
        response = requests.get(url, params=params)
        return response.json()
    
    def get_party_votes_for_session(self, voting_id):
        """Extract party-level voting patterns from a session"""
        session_data = self.get_voting_session_with_votes(voting_id)
        
        party_votes = defaultdict(lambda: {'for': 0, 'against': 0, 'absent': 0, 'abstain': 0})
        
        if session_data['value']:
            votes = session_data['value'][0].get('Stemme', [])
            
            for vote in votes:
                actor = vote.get('Aktør', {})
                party = self.extract_party_from_actor(actor)
                vote_type = vote.get('typeid')
                
                if party:
                    if vote_type == 1:
                        party_votes[party]['for'] += 1
                    elif vote_type == 2:
                        party_votes[party]['against'] += 1
                    elif vote_type == 3:
                        party_votes[party]['absent'] += 1
                    elif vote_type == 4:
                        party_votes[party]['abstain'] += 1
        
        return dict(party_votes)
    
    def extract_party_from_actor(self, actor):
        """Extract party affiliation from actor data"""
        # Party information is typically in the actor's biographical data
        biografi = actor.get('biografi', '')
        
        # Parse party information from biography XML
        # This is a simplified parser - real implementation needs robust XML parsing
        if 'Socialdemokratiet' in biografi or 'socialdemokrat' in biografi.lower():
            return 'S'
        elif 'Venstre' in biografi and 'Danmarks Liberale Parti' in biografi:
            return 'V'
        elif 'Dansk Folkeparti' in biografi:
            return 'DF'
        elif 'Det Konservative Folkeparti' in biografi:
            return 'KF'
        elif 'Socialistisk Folkeparti' in biografi:
            return 'SF'
        elif 'Det Radikale Venstre' in biografi:
            return 'RV'
        elif 'Enhedslisten' in biografi:
            return 'EL'
        elif 'Liberal Alliance' in biografi:
            return 'LA'
        else:
            return 'Unknown'

In [ ]:
analyzer = PartyVotingAnalyzer()
voting_id = 10377  # Recent voting session from API
analyzer.get_voting_session_with_votes(voting_id)
# party_votes = analyzer.get_party_votes_for_session(voting_id)
# Analyze specific voting session


{'odata.metadata': 'https://oda.ft.dk/api/$metadata#Afstemning',
 'value': [{'Stemme': [{'Aktør': {'id': 12,
      'typeid': 5,
      'gruppenavnkort': None,
      'navn': 'Nicolai Wammen',
      'fornavn': 'Nicolai',
      'efternavn': 'Wammen',
      'biografi': '<member><url/><status>1</status><sex>Mand</sex><educationStatistic>LVU</educationStatistic><occupationStatistic>Uden</occupationStatistic><title>Nicolai Wammen (S)</title><firstname>Nicolai</firstname><lastname>Wammen</lastname><profession>Finansminister</profession><party>Socialdemokratiet</party><partyShortname>S</partyShortname><formattedDateLongMonth/><born>07-02-1971</born><died/><pictureMiRes>https://www.ft.dk/-/media/cv/foto/20111/s/nicolai_wammen/nicolai_wammen_pdf,-d-,jpg.ashx</pictureMiRes><pictureHiRes>https://www.ft.dk/-/media/cv/foto/20111/s/nicolai_wammen/nicolai_wammen.zip</pictureHiRes><addresses><address>Finansministeriet, Christiansborg Slotsplads 1 1218&amp;nbsp;København K</address></addresses><phoneFolke

In [6]:
print(party_votes)

{'S': {'for': 18, 'against': 0, 'absent': 14, 'abstain': 0}, 'DF': {'for': 7, 'against': 0, 'absent': 5, 'abstain': 0}, 'SF': {'for': 3, 'against': 1, 'absent': 6, 'abstain': 0}, 'Unknown': {'for': 16, 'against': 0, 'absent': 16, 'abstain': 0}, 'LA': {'for': 1, 'against': 0, 'absent': 2, 'abstain': 0}, 'EL': {'for': 0, 'against': 3, 'absent': 4, 'abstain': 0}, 'KF': {'for': 2, 'against': 0, 'absent': 2, 'abstain': 0}}


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from matplotlib.patches import Rectangle
import numpy as np

class PartyVotingVisualizer:
    def __init__(self, analyzer):
        self.analyzer = analyzer
        
        # Set up plotting style
        plt.style.use('default')
        sns.set_palette("husl")
        
        # Danish party colors (approximate)
        self.party_colors = {
            'S': '#E3515D',    # Social Democrats - Red
            'V': '#0059A3',    # Venstre - Blue  
            'DF': '#E6D845',   # Danish People's Party - Yellow
            'KF': '#0C4F60',   # Conservatives - Dark Blue
            'SF': '#9C1F2F',   # Socialist People's Party - Dark Red
            'RV': '#733280',   # Radical Left - Purple
            'EL': '#E07EA8',   # Red-Green Alliance - Pink
            'LA': '#1B365D',   # Liberal Alliance - Dark Blue
        }
    
    def plot_party_cohesion_timeline(self, cohesion_data, party_name, save_path=None):
        """Plot party cohesion over time"""
        
        if not cohesion_data:
            print(f"No cohesion data available for {party_name}")
            return
        
        dates = [datetime.fromisoformat(entry['date'].replace('Z', '+00:00')) for entry in cohesion_data if entry.get('date')]
        rice_indices = [entry['rice_index'] for entry in cohesion_data if entry.get('rice_index') is not None]
        
        if not dates or not rice_indices:
            print(f"Insufficient data for plotting {party_name} cohesion")
            return
        
        plt.figure(figsize=(12, 6))
        plt.plot(dates, rice_indices, 
                marker='o', markersize=4, 
                color=self.party_colors.get(party_name, 'gray'),
                linewidth=2, alpha=0.8)
        
        plt.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='High Cohesion Threshold')
        plt.axhline(y=60, color='orange', linestyle='--', alpha=0.7, label='Medium Cohesion Threshold')
        
        plt.title(f'{party_name} Party Cohesion Over Time (Rice Index)', fontsize=14, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Rice Index (%)', fontsize=12)
        plt.ylim(0, 100)
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    def plot_agreement_matrix_heatmap(self, agreement_matrix, save_path=None):
        """Plot inter-party agreement matrix as heatmap"""
        
        plt.figure(figsize=(10, 8))
        
        # Create custom colormap
        mask = np.diag(np.ones(len(agreement_matrix)))  # Mask diagonal
        
        sns.heatmap(agreement_matrix, 
                   annot=True, 
                   fmt='.1f',
                   cmap='RdYlBu_r',
                   center=50,
                   square=True,
                   mask=mask,
                   cbar_kws={'label': 'Agreement Rate (%)'})
        
        plt.title('Inter-Party Agreement Matrix', fontsize=14, fontweight='bold')
        plt.xlabel('Party', fontsize=12)
        plt.ylabel('Party', fontsize=12)
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    def plot_voting_network(self, agreement_matrix, threshold=70, save_path=None):
        """Plot party relationships as network graph"""
        
        plt.figure(figsize=(12, 10))
        
        # Create network graph
        G = nx.Graph()
        parties = list(agreement_matrix.index)
        
        # Add nodes
        for party in parties:
            G.add_node(party)
        
        # Add edges for strong agreements
        for i, party_a in enumerate(parties):
            for j, party_b in enumerate(parties):
                if i < j:  # Avoid duplicate edges
                    agreement = agreement_matrix.iloc[i, j]
                    if agreement >= threshold:
                        G.add_edge(party_a, party_b, weight=agreement)
        
        # Set up layout
        pos = nx.spring_layout(G, k=3, iterations=50)
        
        # Draw network
        # Draw edges with thickness based on agreement strength
        edges = G.edges(data=True)
        for (u, v, d) in edges:
            weight = d['weight']
            plt.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], 
                    'gray', alpha=0.6, linewidth=(weight-threshold)/10+1)
        
        # Draw nodes
        for party in parties:
            x, y = pos[party]
            color = self.party_colors.get(party, 'gray')
            plt.scatter(x, y, s=2000, c=color, alpha=0.8, edgecolors='black', linewidth=2)
            plt.text(x, y, party, ha='center', va='center', fontsize=12, fontweight='bold', color='white')
        
        plt.title(f'Party Agreement Network (e{threshold}% agreement)', fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    def plot_coalition_performance(self, coalition_comparison, save_path=None):
        """Plot coalition performance comparison"""
        
        coalitions = list(coalition_comparison.keys())
        cohesion_rates = [coalition_comparison[c]['cohesion_rate'] for c in coalitions]
        stress_points = [coalition_comparison[c]['stress_points'] for c in coalitions]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Cohesion rates
        bars1 = ax1.bar(coalitions, cohesion_rates, 
                       color=['#E3515D', '#FF9999', '#9999FF'], alpha=0.8)
        ax1.set_title('Coalition Cohesion Rates', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Cohesion Rate (%)', fontsize=12)
        ax1.set_ylim(0, 100)
        ax1.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='High Cohesion')
        
        # Add value labels on bars
        for bar, value in zip(bars1, cohesion_rates):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{value:.1f}%', ha='center', va='bottom', fontweight='bold')
        
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Stress points
        bars2 = ax2.bar(coalitions, stress_points, 
                       color=['#FF6B6B', '#FFB366', '#66B3FF'], alpha=0.8)
        ax2.set_title('Coalition Stress Points', fontsize=14, fontweight='bold')
        ax2.set_ylabel('Number of Disagreements', fontsize=12)
        
        # Add value labels on bars
        for bar, value in zip(bars2, stress_points):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(value), ha='center', va='bottom', fontweight='bold')
        
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    def plot_rebellion_analysis(self, rebellion_data, party_name, save_path=None):
        """Plot party rebellion patterns by topic"""
        
        if not rebellion_data:
            print(f"No rebellion data available for {party_name}")
            return
        
        # Count rebellions by topic
        topic_counts = defaultdict(int)
        for mp_rebellions in rebellion_data.values():
            for rebellion in mp_rebellions:
                topic_counts[rebellion['topic']] += 1
        
        topics = list(topic_counts.keys())
        counts = list(topic_counts.values())
        
        plt.figure(figsize=(12, 8))
        
        # Create horizontal bar chart
        bars = plt.barh(topics, counts, color=self.party_colors.get(party_name, 'gray'), alpha=0.8)
        
        plt.title(f'{party_name} Party Rebellions by Topic', fontsize=14, fontweight='bold')
        plt.xlabel('Number of Rebellious Votes', fontsize=12)
        plt.ylabel('Policy Topic', fontsize=12)
        
        # Add value labels
        for bar, count in zip(bars, counts):
            plt.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                    str(count), va='center', fontweight='bold')
        
        plt.grid(True, alpha=0.3, axis='x')
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    def plot_cross_party_collaboration(self, collaboration_data, save_path=None):
        """Plot cross-party collaboration patterns"""
        
        if not collaboration_data:
            print("No collaboration data available")
            return
        
        # Extract collaboration info
        coalition_sizes = []
        coalition_labels = []
        topic_data = defaultdict(int)
        
        for coalition, info in collaboration_data.items():
            coalition_sizes.append(info['frequency'])
            coalition_labels.append(' + '.join(info['parties'][:3]))  # Limit label length
            
            for topic, count in info['main_topics'].items():
                topic_data[topic] += count
        
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))
        
        # Coalition frequency
        bars1 = ax1.bar(range(len(coalition_labels)), coalition_sizes, 
                       color='lightblue', alpha=0.8, edgecolor='navy')
        ax1.set_title('Cross-Party Coalition Frequency', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Coalition', fontsize=12)
        ax1.set_ylabel('Number of Joint Votes', fontsize=12)
        ax1.set_xticks(range(len(coalition_labels)))
        ax1.set_xticklabels(coalition_labels, rotation=45, ha='right')
        
        for bar, size in zip(bars1, coalition_sizes):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(size), ha='center', va='bottom', fontweight='bold')
        
        # Topic distribution
        topics = list(topic_data.keys())
        topic_counts = list(topic_data.values())
        
        wedges, texts, autotexts = ax2.pie(topic_counts, labels=topics, autopct='%1.1f%%',
                                          startangle=90, colors=plt.cm.Set3(np.linspace(0, 1, len(topics))))
        ax2.set_title('Cross-Party Collaboration by Topic', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()
    
    def create_comprehensive_dashboard(self, party_data, save_path=None):
        """Create comprehensive party analysis dashboard"""
        
        fig = plt.figure(figsize=(20, 12))
        gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
        
        # Dashboard components would go here
        # This is a framework for a comprehensive visualization
        
        plt.suptitle('Danish Parliamentary Party Analysis Dashboard', 
                    fontsize=16, fontweight='bold', y=0.98)
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

In [4]:
#THIS IS THE ONE WE WANT TO USE, BUT WE NEED TO DEFINE A BUNCH OF OTHER FUNCTIONS FIRST.


# Initialize visualizer
visualizer = PartyVotingVisualizer(analyzer)

# Example 1: Plot party cohesion timeline
cohesion_analyzer = PartyCohesionAnalyzer(analyzer)
s_timeline = cohesion_analyzer.analyze_party_cohesion_over_time('S', '2024-01-01', '2024-12-31')
visualizer.plot_party_cohesion_timeline(s_timeline, 'S', 'social_democrats_cohesion.png')

# Example 2: Agreement matrix heatmap
inter_party_analyzer = InterPartyAnalyzer(analyzer)
recent_sessions = list(range(10360, 10378))
agreement_matrix = inter_party_analyzer.calculate_agreement_matrix(recent_sessions)
visualizer.plot_agreement_matrix_heatmap(agreement_matrix, 'party_agreement_heatmap.png')

# Example 3: Voting network visualization
visualizer.plot_voting_network(agreement_matrix, threshold=65, save_path='party_network.png')

# Example 4: Coalition performance comparison
coalition_analyzer = CoalitionAnalyzer(analyzer)
all_coalitions = ['current_government', 'red_bloc', 'blue_bloc']
comparison = coalition_analyzer.compare_coalition_performance(all_coalitions, recent_sessions)
visualizer.plot_coalition_performance(comparison, 'coalition_performance.png')

NameError: name 'analyzer' is not defined

In [13]:
import numpy as np

class PartyCohesionAnalyzer:
    def __init__(self, analyzer):
        self.analyzer = analyzer
    
    def calculate_rice_index(self, party_votes):
        """
        Calculate Rice Index for party cohesion
        Rice Index = (|Yes - No|) / (Yes + No) * 100
        Scale: 0-100, where 100 = perfect unity
        """
        rice_scores = {}
        
        for party, votes in party_votes.items():
            yes_votes = votes['for']
            no_votes = votes['against']
            total_votes = yes_votes + no_votes
            
            if total_votes > 0:
                rice_index = abs(yes_votes - no_votes) / total_votes * 100
                rice_scores[party] = rice_index
            else:
                rice_scores[party] = None
        
        return rice_scores
    
    def calculate_agreement_index(self, party_votes):
        """
        Alternative cohesion measure: Agreement Index
        Agreement Index = (Max(Yes, No)) / (Yes + No) * 100
        """
        agreement_scores = {}
        
        for party, votes in party_votes.items():
            yes_votes = votes['for']
            no_votes = votes['against']
            total_votes = yes_votes + no_votes
            
            if total_votes > 0:
                agreement_index = max(yes_votes, no_votes) / total_votes * 100
                agreement_scores[party] = agreement_index
            else:
                agreement_scores[party] = None
        
        return agreement_scores
    
    def analyze_party_cohesion_over_time(self, party, start_date='2024-01-01', end_date='2024-12-31'):
        """Analyze party cohesion trends over time period"""
        
        # Get all voting sessions in time period
        params = {
            '$expand': 'Møde,Stemme/Aktør',
            '$filter': f"Møde/dato ge datetime'{start_date}' and Møde/dato le datetime'{end_date}'",
            '$orderby': 'Møde/dato'
        }
        
        cohesion_timeline = []
        
        # This is a simplified example - real implementation needs pagination
        url = f"{self.analyzer.base_url}Afstemning"
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            sessions = response.json().get('value', [])
            
            for session in sessions:
                party_votes = self.analyzer.get_party_votes_for_session(session['id'])
                
                if party in party_votes:
                    rice_index = self.calculate_rice_index({party: party_votes[party]})[party]
                    
                    cohesion_timeline.append({
                        'date': session.get('Møde', {}).get('dato'),
                        'voting_id': session['id'],
                        'rice_index': rice_index,
                        'votes': party_votes[party]
                    })
        
        return cohesion_timeline

In [14]:
# Initialize analyzers
analyzer = PartyVotingAnalyzer()
cohesion_analyzer = PartyCohesionAnalyzer(analyzer)

# Analyze specific voting session
voting_id = 10377  # Recent voting session from API
party_votes = analyzer.get_party_votes_for_session(voting_id)

# Calculate cohesion scores
rice_scores = cohesion_analyzer.calculate_rice_index(party_votes)
agreement_scores = cohesion_analyzer.calculate_agreement_index(party_votes)

# Display results
for party in rice_scores:
    print(f"{party}: Rice Index = {rice_scores[party]:.1f}%, Agreement Index = {agreement_scores[party]:.1f}%")

S: Rice Index = 100.0%, Agreement Index = 100.0%
DF: Rice Index = 100.0%, Agreement Index = 100.0%
SF: Rice Index = 50.0%, Agreement Index = 75.0%
Unknown: Rice Index = 100.0%, Agreement Index = 100.0%
LA: Rice Index = 100.0%, Agreement Index = 100.0%
EL: Rice Index = 100.0%, Agreement Index = 100.0%
KF: Rice Index = 100.0%, Agreement Index = 100.0%


In [16]:
import itertools
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import jaccard_score

class InterPartyAnalyzer:
    def __init__(self, analyzer):
        self.analyzer = analyzer
    
    def calculate_agreement_matrix(self, voting_sessions):
        """Calculate pairwise agreement matrix between parties"""
        
        # Collect all party positions across voting sessions
        party_positions = defaultdict(list)
        
        for session_id in voting_sessions:
            party_votes = self.analyzer.get_party_votes_for_session(session_id)
            
            # Determine party position (simplified: majority vote wins)
            for party, votes in party_votes.items():
                total_votes = votes['for'] + votes['against']
                if total_votes > 0:
                    position = 1 if votes['for'] > votes['against'] else 0
                    party_positions[party].append(position)
                else:
                    party_positions[party].append(None)  # No clear position
        
        # Calculate pairwise agreements
        parties = list(party_positions.keys())
        n_parties = len(parties)
        agreement_matrix = np.zeros((n_parties, n_parties))
        
        for i, party_a in enumerate(parties):
            for j, party_b in enumerate(parties):
                if i != j:
                    # Calculate agreement percentage
                    positions_a = party_positions[party_a]
                    positions_b = party_positions[party_b]
                    
                    agreements = 0
                    valid_comparisons = 0
                    
                    for pos_a, pos_b in zip(positions_a, positions_b):
                        if pos_a is not None and pos_b is not None:
                            valid_comparisons += 1
                            if pos_a == pos_b:
                                agreements += 1
                    
                    if valid_comparisons > 0:
                        agreement_matrix[i][j] = agreements / valid_comparisons * 100
                    else:
                        agreement_matrix[i][j] = 0
                else:
                    agreement_matrix[i][j] = 100  # Party agrees with itself
        
        return pd.DataFrame(agreement_matrix, index=parties, columns=parties)
    
    def find_voting_blocs(self, agreement_matrix, threshold=70):
        """Identify voting blocs based on agreement threshold"""
        
        blocs = []
        parties = list(agreement_matrix.index)
        unassigned = set(parties)
        
        while unassigned:
            # Start with an unassigned party
            seed_party = next(iter(unassigned))
            bloc = {seed_party}
            unassigned.remove(seed_party)
            
            # Find parties that agree with this bloc above threshold
            for party in list(unassigned):
                # Check agreement with all parties in current bloc
                agreements = [agreement_matrix.loc[party, bloc_member] 
                            for bloc_member in bloc]
                
                if all(agreement >= threshold for agreement in agreements):
                    bloc.add(party)
                    unassigned.remove(party)
            
            blocs.append(bloc)
        
        return blocs
    
    def calculate_polarization_index(self, agreement_matrix):
        """Calculate overall system polarization"""
        
        # Average inter-party agreement (excluding self-agreement)
        n = len(agreement_matrix)
        total_agreement = 0
        count = 0
        
        for i in range(n):
            for j in range(n):
                if i != j:
                    total_agreement += agreement_matrix.iloc[i, j]
                    count += 1
        
        avg_agreement = total_agreement / count if count > 0 else 0
        
        # Polarization is inverse of agreement (scaled)
        polarization_index = (100 - avg_agreement) / 100
        
        return {
            'average_agreement': avg_agreement,
            'polarization_index': polarization_index,
            'interpretation': self.interpret_polarization(polarization_index)
        }
    
    def interpret_polarization(self, index):
        """Provide interpretation of polarization index"""
        if index < 0.3:
            return "Low polarization - high cross-party consensus"
        elif index < 0.5:
            return "Moderate polarization - some partisan divisions"
        elif index < 0.7:
            return "High polarization - significant partisan conflict"
        else:
            return "Extreme polarization - minimal cross-party agreement"

In [17]:
# Analyze recent voting sessions
recent_sessions = [10377, 10376, 10375, 10374, 10373]  # Recent session IDs

inter_party_analyzer = InterPartyAnalyzer(analyzer)
agreement_matrix = inter_party_analyzer.calculate_agreement_matrix(recent_sessions)

print("Inter-party Agreement Matrix:")
print(agreement_matrix.round(1))

# Find voting blocs
blocs = inter_party_analyzer.find_voting_blocs(agreement_matrix)
print(f"\nVoting Blocs (>70% agreement):")
for i, bloc in enumerate(blocs, 1):
    print(f"Bloc {i}: {', '.join(bloc)}")

# Calculate polarization
polarization = inter_party_analyzer.calculate_polarization_index(agreement_matrix)
print(f"\nSystem Polarization: {polarization['polarization_index']:.3f}")
print(f"Interpretation: {polarization['interpretation']}")

Inter-party Agreement Matrix:
             S     DF     SF  Unknown     LA     EL     KF
S        100.0   60.0   80.0    100.0   60.0   40.0  100.0
DF        60.0  100.0   80.0     60.0  100.0   80.0   60.0
SF        80.0   80.0  100.0     80.0   80.0   60.0   80.0
Unknown  100.0   60.0   80.0    100.0   60.0   40.0  100.0
LA        60.0  100.0   80.0     60.0  100.0   80.0   60.0
EL        40.0   80.0   60.0     40.0   80.0  100.0   40.0
KF       100.0   60.0   80.0    100.0   60.0   40.0  100.0

Voting Blocs (>70% agreement):
Bloc 1: DF, SF, LA
Bloc 2: S, Unknown, KF
Bloc 3: EL

System Polarization: 0.286
Interpretation: Low polarization - high cross-party consensus


In [18]:
class CoalitionAnalyzer:
    def __init__(self, analyzer):
        self.analyzer = analyzer
        
        # Define current and historical coalitions
        self.coalitions = {
            'current_government': ['S', 'V'],  # Example: Social Democrats + Liberals
            'red_bloc': ['S', 'SF', 'RV', 'EL'],  # Left-wing parties
            'blue_bloc': ['V', 'KF', 'DF', 'LA'],  # Right-wing parties
        }
    
    def analyze_coalition_cohesion(self, coalition_name, voting_sessions):
        """Analyze voting cohesion within a coalition"""
        
        if coalition_name not in self.coalitions:
            raise ValueError(f"Unknown coalition: {coalition_name}")
        
        coalition_parties = self.coalitions[coalition_name]
        coalition_agreements = []
        
        for session_id in voting_sessions:
            party_votes = self.analyzer.get_party_votes_for_session(session_id)
            
            # Get positions of coalition parties
            coalition_positions = {}
            for party in coalition_parties:
                if party in party_votes:
                    votes = party_votes[party]
                    total_votes = votes['for'] + votes['against']
                    
                    if total_votes > 0:
                        position = 1 if votes['for'] > votes['against'] else 0
                        coalition_positions[party] = position
            
            # Calculate agreement within coalition
            if len(coalition_positions) >= 2:
                positions = list(coalition_positions.values())
                agreement = len(set(positions)) == 1  # All parties agree
                coalition_agreements.append({
                    'session_id': session_id,
                    'agreement': agreement,
                    'positions': coalition_positions
                })
        
        cohesion_rate = sum(1 for x in coalition_agreements if x['agreement']) / len(coalition_agreements) * 100
        
        return {
            'coalition': coalition_name,
            'parties': coalition_parties,
            'cohesion_rate': cohesion_rate,
            'agreements': coalition_agreements
        }
    
    def identify_coalition_stress_points(self, coalition_analysis):
        """Identify votes where coalition parties disagreed"""
        
        stress_points = []
        
        for agreement in coalition_analysis['agreements']:
            if not agreement['agreement']:
                # Get detailed voting information
                session_id = agreement['session_id']
                session_data = self.analyzer.get_voting_session_with_votes(session_id)
                
                if session_data['value']:
                    session_info = session_data['value'][0]
                    stress_points.append({
                        'session_id': session_id,
                        'conclusion': session_info.get('konklusion', 'No conclusion'),
                        'disagreeing_parties': agreement['positions'],
                        'meeting_date': session_info.get('Møde', {}).get('dato')
                    })
        
        return stress_points
    
    def compare_coalition_performance(self, coalitions, voting_sessions):
        """Compare performance across multiple coalitions"""
        
        comparison = {}
        
        for coalition_name in coalitions:
            analysis = self.analyze_coalition_cohesion(coalition_name, voting_sessions)
            comparison[coalition_name] = {
                'cohesion_rate': analysis['cohesion_rate'],
                'party_count': len(analysis['parties']),
                'stress_points': len(self.identify_coalition_stress_points(analysis))
            }
        
        return comparison

In [19]:
# Initialize coalition analyzer
coalition_analyzer = CoalitionAnalyzer(analyzer)

# Analyze current government coalition
recent_sessions = [10377, 10376, 10375, 10374, 10373]
government_analysis = coalition_analyzer.analyze_coalition_cohesion('current_government', recent_sessions)

print(f"Government Coalition Cohesion: {government_analysis['cohesion_rate']:.1f}%")

# Identify stress points
stress_points = coalition_analyzer.identify_coalition_stress_points(government_analysis)
print(f"Coalition disagreements: {len(stress_points)} out of {len(recent_sessions)} votes")

# Compare all coalitions
all_coalitions = ['current_government', 'red_bloc', 'blue_bloc']
comparison = coalition_analyzer.compare_coalition_performance(all_coalitions, recent_sessions)

for coalition, metrics in comparison.items():
    print(f"{coalition}: {metrics['cohesion_rate']:.1f}% cohesion, {metrics['stress_points']} stress points")

ZeroDivisionError: division by zero

In [1]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

class PartyEvolutionAnalyzer:
    def __init__(self, analyzer):
        self.analyzer = analyzer
    
    def track_party_positions_over_time(self, party, topic_keywords, months_back=12):
        """Track a party's voting positions on specific topics over time"""
        
        end_date = datetime.now()
        start_date = end_date - timedelta(days=months_back * 30)
        
        # Get voting sessions with case information
        params = {
            '$expand': 'Stemme/Aktör,Møde,Sag',
            '$filter': f"Møde/dato ge datetime'{start_date.isoformat()}' and Møde/dato le datetime'{end_date.isoformat()}'",
            '$orderby': 'Møde/dato',
            '$top': 100  # API limit
        }
        
        timeline = []
        
        # This example shows the concept - real implementation needs pagination
        url = f"{self.analyzer.base_url}Afstemning"
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            sessions = response.json().get('value', [])
            
            for session in sessions:
                # Check if session relates to topics of interest
                sag_info = session.get('Sag', {})
                sag_title = sag_info.get('titel', '').lower()
                
                if any(keyword.lower() in sag_title for keyword in topic_keywords):
                    party_votes = self.analyzer.get_party_votes_for_session(session['id'])
                    
                    if party in party_votes:
                        votes = party_votes[party]
                        total_votes = votes['for'] + votes['against']
                        
                        if total_votes > 0:
                            support_rate = votes['for'] / total_votes
                            
                            timeline.append({
                                'date': session.get('Møde', {}).get('dato'),
                                'session_id': session['id'],
                                'case_title': sag_info.get('titel'),
                                'support_rate': support_rate,
                                'votes_for': votes['for'],
                                'votes_against': votes['against'],
                                'abstentions': votes['abstain'],
                                'absent': votes['absent']
                            })
        
        return sorted(timeline, key=lambda x: x['date'])
    
    def calculate_position_drift(self, timeline, window_size=5):
        """Calculate how much a party's position has drifted over time"""
        
        if len(timeline) < window_size * 2:
            return None
        
        # Early period average
        early_positions = [entry['support_rate'] for entry in timeline[:window_size]]
        early_avg = sum(early_positions) / len(early_positions)
        
        # Recent period average
        recent_positions = [entry['support_rate'] for entry in timeline[-window_size:]]
        recent_avg = sum(recent_positions) / len(recent_positions)
        
        drift = recent_avg - early_avg
        
        return {
            'early_support': early_avg,
            'recent_support': recent_avg,
            'drift': drift,
            'interpretation': self.interpret_drift(drift)
        }
    
    def interpret_drift(self, drift):
        """Interpret the meaning of position drift"""
        if abs(drift) < 0.1:
            return "Stable position"
        elif drift > 0.3:
            return "Significant shift toward support"
        elif drift > 0.1:
            return "Moderate shift toward support"
        elif drift < -0.3:
            return "Significant shift toward opposition"
        elif drift < -0.1:
            return "Moderate shift toward opposition"
        else:
            return "Minor position adjustment"
    
    def detect_position_reversals(self, timeline, threshold=0.5):
        """Identify cases where a party completely reversed position"""
        
        reversals = []
        
        for i in range(1, len(timeline)):
            current = timeline[i]['support_rate']
            previous = timeline[i-1]['support_rate']
            
            # Check for reversal (from support to opposition or vice versa)
            if (previous >= threshold and current < (1 - threshold)) or \
               (previous < (1 - threshold) and current >= threshold):
                
                reversals.append({
                    'from_date': timeline[i-1]['date'],
                    'to_date': timeline[i]['date'],
                    'from_case': timeline[i-1]['case_title'],
                    'to_case': timeline[i]['case_title'],
                    'from_support': previous,
                    'to_support': current
                })
        
        return reversals

In [21]:
# Initialize evolution analyzer
evolution_analyzer = PartyEvolutionAnalyzer(analyzer)

# Track Social Democrats' position on climate issues
climate_keywords = ['klima', 'miljø', 'grøn', 'bæredygtighed', 'CO2']
s_climate_timeline = evolution_analyzer.track_party_positions_over_time('S', climate_keywords, months_back=24)

print(f"Found {len(s_climate_timeline)} climate-related votes for Social Democrats")

# Analyze position drift
drift_analysis = evolution_analyzer.calculate_position_drift(s_climate_timeline)
if drift_analysis:
    print(f"Position drift: {drift_analysis['drift']:.3f} ({drift_analysis['interpretation']})")
    print(f"Early support rate: {drift_analysis['early_support']:.1%}")
    print(f"Recent support rate: {drift_analysis['recent_support']:.1%}")

# Detect reversals
reversals = evolution_analyzer.detect_position_reversals(s_climate_timeline)
print(f"Position reversals detected: {len(reversals)}")

for reversal in reversals:
    print(f"  {reversal['from_date']} � {reversal['to_date']}: {reversal['from_support']:.1%} � {reversal['to_support']:.1%}")

Found 0 climate-related votes for Social Democrats
Position reversals detected: 0
